In [1]:
%pip install pandas sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

DB_USER = os.getenv('POSTGRES_USER', 'joomlauser')
DB_PASS = os.getenv('POSTGRES_PASSWORD', 'joomlapassword')
DB_HOST = 'database'
DB_PORT = '5432'
DB_NAME = os.getenv('POSTGRES_DB', 'joomladb')

try:
    engine = create_engine(f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
    
    # Consulta para obtener las tablas de la base de datos
    query = text("SELECT table_name FROM information_schema.tables WHERE table_schema='public';")
    
    with engine.connect() as conn:
        result = conn.execute(query)
        df_tables = pd.DataFrame(result.fetchall(), columns=result.keys())
    
    print(" Conexión exitosa a la base de datos PostgreSQL!")
    print(f"Se encontraron {len(df_tables)} tablas creadas por Joomla.")
    print(df_tables.head(15))
except Exception as e:
    print(f" Error de conexión: {e}")

 Conexión exitosa a la base de datos PostgreSQL!
Se encontraron 152 tablas creadas por Joomla.
                      table_name
0        fl3x2_action_log_config
1              fl3x2_action_logs
2   fl3x2_action_logs_extensions
3        fl3x2_action_logs_users
4                   fl3x2_assets
5             fl3x2_associations
6           fl3x2_banner_clients
7            fl3x2_banner_tracks
8                  fl3x2_banners
9               fl3x2_categories
10         fl3x2_contact_details
11                 fl3x2_content
12       fl3x2_content_frontpage
13          fl3x2_content_rating
14           fl3x2_content_types


In [12]:
from sqlalchemy import text

with engine.connect() as conn:

    print("=== EXTENSIONES ===")
    resultado1 = conn.execute(text("""
        SELECT
            type AS tipo,
            COUNT(*) AS cantidad
        FROM fl3x2_extensions
        GROUP BY type
        ORDER BY cantidad DESC;
    """)).fetchall()

    for fila in resultado1:
        print(fila)

    print("\n=== USUARIOS POR GRUPO ===")
    resultado2 = conn.execute(text("""
        SELECT
            ug.title AS grupo,
            COUNT(u.id) AS usuarios
        FROM fl3x2_usergroups ug
        LEFT JOIN fl3x2_user_usergroup_map uum
            ON ug.id = uum.group_id
        LEFT JOIN fl3x2_users u
            ON u.id = uum.user_id
        GROUP BY ug.id, ug.title
        ORDER BY usuarios DESC;
    """)).fetchall()

    for fila in resultado2:
        print(fila)

=== EXTENSIONES ===
('plugin', 156)
('module', 49)
('component', 36)
('language', 3)
('template', 3)
('library', 2)
('package', 1)
('file', 1)

=== USUARIOS POR GRUPO ===
('Super Users', 1)
('Guest', 0)
('Administrator', 0)
('Public', 0)
('Publisher', 0)
('Editor', 0)
('Registered', 0)
('Manager', 0)
('Author', 0)
